In [1]:
gg_colab = True
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install networkx==3.5 --force-reinstall
!pip install setfit
!pip install underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.4 MB/s eta 0:00:00
  Attempting uninstall: networkx
    Found existing installation: networkx 3.6.1
    Uninstalling networkx-3.6.1:
      Successfully uninstalled networkx-3.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.2 MB/s eta 0:00:00


In [3]:
if gg_colab:
  GG_COLAB = "/content/drive/MyDrive/MXH"
  TRIPLES_JSON = f'{GG_COLAB}/data/triples_for_RE.json'
  WIKI_ENRICHMENT = f'{GG_COLAB}/data/wiki_enrichment.jsonl'
  BIPARTITE_JSON = f'{GG_COLAB}/data/B.json'
  RE_MODEL = f'{GG_COLAB}/data/re_model'
  # RE_RES = f'{GG_COLAB}/data/re_results.jsonl'
else:
  TRIPLES = './data/triples_for_RE.json'
  WIKI_ENRICHMENT = './data/wiki_enrichment.jsonl'
  BIPARTITE_JSON = './data/vn_bipartite_graph.json'





In [4]:
import re

import json
from setfit import Trainer
import random
from collections import defaultdict
import underthesea
import unicodedata
import networkx as nx
print(nx.__version__) # 3.5
from networkx.readwrite import json_graph
from underthesea import ner as uts_ner
from underthesea import pos_tag as uts_pos_tag
from underthesea import word_tokenize as uts_word_tokenize
from collections import Counter, defaultdict,OrderedDict
from setfit import SetFitModel
import itertools


3.5


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
def load_graph(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return json_graph.node_link_graph(data,link="edges" )

def load_bipartite_graph_and_nodes(B):
    # Tách danh sách ACTORS & MOVIES
    person_list = []
    film_list = []
    for node, attrs in B.nodes(data=True):
        ntype = attrs.get("type")
        if ntype == "person":
            person_list.append(node)
        elif ntype == "film":
            film_list.append(node)

    return B, person_list, film_list



B = load_graph(BIPARTITE_JSON)
B, person_list, film_list = load_bipartite_graph_and_nodes(B)

def load_jsonl_to_dict(path):
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            name = obj.get("name")
            if name:
                result[name] = obj
    return result

wiki_enrich = load_jsonl_to_dict(WIKI_ENRICHMENT)



# loại bỏ dấu câu trước khi xử lý văn bản
RE_PUNCT = re.compile(r"[\u2000-\u206F\u2E00-\u2E7F\'\"“”‘’!#$%&()*+,\-./:;<=>?@\[\]^_`{|}~…—·]+")
# chuẩn hóa khoảng trắng
RE_SPACE = re.compile(r"\s+")
# vào str -> trả str
def split_text_into_sentences(text):
    return underthesea.sent_tokenize(text)


def remove_text_in_parentheses(text):   # xóa text trong ngoặc
    cleaned_text = re.sub(r"\s*\(.*?\)", "", text)  # kết quả sau khi xóa
    return cleaned_text
print(remove_text_in_parentheses('Hoa Mặt Trời (phim truyền hình)')) # ==> Hoa Mặt Trời
# ************************************ Dùng cho văn bản dài

def normalize_text_for_nlp(text):
    """
    Chuẩn hóa toàn diện cho text:
    - Unicode NFKC
    - lowercase
    - remove punctuation
    - remove extra spaces
    - strip
    """
    if not text:
        return ""

    # Normalize Unicode + lowercase
    text = unicodedata.normalize("NFKC", str(text)).lower()

    # Remove punctuation
    text = RE_PUNCT.sub(" ", text)

    # Remove extra spaces
    text = RE_SPACE.sub(" ", text)

    return text.strip()

print(normalize_text_for_nlp('Hoa Mặt Trời')) # ==> hoa mặt trời


import re
import unicodedata

def normalize_entity_name(x):
    # Ninh Dương Lan Ngọc . ==> Ninh Dương Lan Ngọc
    """Chuẩn hoá tên entity: unicode, xoá khoảng trắng, gom space, bỏ dấu câu đầu/cuối."""
    if not isinstance(x, str):
        x = str(x)

    # Chuẩn hoá unicode + bỏ khoảng trắng đầu/cuối
    x = unicodedata.normalize("NFKC", x).strip()
    if not x:
        return ""

    # Nếu chuỗi chỉ toàn ký tự đặc biệt / số → loại bỏ
    if re.fullmatch(r"[\W\d_]+", x):
        return ""

    # Gom nhiều khoảng trắng thành 1
    x = re.sub(r"\s+", " ", x).strip()

    # Hàm kiểm tra ký tự có phải dấu câu (Unicode) không
    # Ví dụ: ., , : ; … “ ” !
    def _is_punct(ch):
        return unicodedata.category(ch).startswith("P")

    # Bỏ dấu câu ở đầu chuỗi
    start = 0
    while start < len(x) and _is_punct(x[start]):
        start += 1

    # Bỏ dấu câu ở cuối chuỗi
    end = len(x) - 1
    while end >= start and _is_punct(x[end]):
        end -= 1

    # Lấy phần còn lại
    x = x[start:end+1].strip()

    # Gom space lại lần cuối
    x = re.sub(r"\s+", " ", x).strip()
    return x


print(normalize_entity_name('Ninh Dương Lan Ngọc .'))



def normalize_type(t, default):
    """Chuẩn hóa type: viết hoa chữ đầu."""
    if not t: return default
    t = str(t).strip()
    if not t: return default
    return t[0].upper() + t[1:]

def norm(s):
    return unicodedata.normalize("NFC", s.strip()).lower()


# Normalize entity ⇒ trả về PER, FILM để khớp với NER combine

def normalize_entity(entity_text, person_list, film_list, wiki_enrich=None):
    if not entity_text:
        return "", "UNK"

    t = entity_text.strip()
    tl = norm(t)

    for p in person_list:
        if tl == norm(p):
            return p, "PER"

    for f in film_list:
        if tl == norm(f):
            return f, "FILM"

    # khớp fuzzy vào wiki
    if wiki_enrich:
        for w in wiki_enrich:
            if tl == norm(w):
                return w, "UNK"

    return t, "UNK"



# NER
# B-XXX = Begin entity (bắt đầu 1 thực thể)
# I-XXX = Inside entity (các từ tiếp theo trong cùng thực thể)
# ====================================

# lọc khỏi vb tập từ ko quan trọng
DEFAULT_STOPWORDS = {
    "và", "là", "của", "cho", "với", "trong", "một", "những", "các", "được",
    "đó", "này", "khi", "đã", "tại", "về", "như", "vẫn", "để", "cũng", "bị",
    "ra", "theo", "vào", "hay", "nhưng", "vì", "do", "nên", "còn", "thì"
}


# =====================================================================
# LAYER 1: BASE NER (UNDERTHESEA)

def ner_raw_underthesea(text):
    """
    Tầng 1 — chạy NER gốc từ Underthesea.
    Output: list [(token, ner_tag)]


    - PER → B-PER
    - ORG → B-ORG
    - LOC → B-LOC
    - O  → O
    - B-xxx / I-xxx giữ nguyên

    ============================================================
    Chạy NER bằng underthesea.
    Input: text (str)
    Output: List[Tuple[str, str]] ==> list [(token, tag)] như underthesea trả về (chưa nhóm BIO)
    note: underthesea.ner trả về list (word, tag) với tag có thể là 'B-ORG', 'I-ORG', 'O', ...
    """
    if not text or not text.strip():
        return []
    # underthesea.ner yêu cầu đầu vào là string và sẽ tự tách token
    ner_out = uts_ner(text)
    # Đảm bảo trả về list of (token, tag) và loại token rỗng
    clean_output = []

    # (word, pos_tag, chunk_tag, ner_tag)
    for item in ner_out:
        # Underthesea format: (word, pos, chunk, ner)
        if len(item) >= 4:
            word, pos, chunk, ner_tag = item[0], item[1], item[2], item[3]
        else:
            continue

        token = str(word).strip()
        tag = str(ner_tag).strip().upper()

        if not token:
            continue

        # Giữ nguyên tag từ Underthesea
        # O, B-LOC, I-LOC, B-PER, I-PER, B-ORG, I-ORG
        bio_tag = tag if tag else "O"

        clean_output.append((token, bio_tag))

    # do underthesea nghĩ rằng phim là loc => fix
    MEDIA_PREFIX = {"phim", "truyện", "bài", "bài hát", "ca khúc", "phim ảnh", "bộ phim"}

    fixed = []
    for w, t in clean_output:
        if w.lower() in MEDIA_PREFIX and t.startswith("B-LOC"):
            fixed.append((w, "O"))
        else:
            fixed.append((w, t))
    return fixed

def canonical_title(name: str) -> str:
    """
    Chuẩn hoá.
    Xoá tất cả mọi thứ trong ngoặc (bao gồm cả ngoặc và nội dung bên trong).
    """
    # raw = name

    # Xoá tất cả mọi thứ trong ngoặc (bao gồm cả ngoặc)
    name = re.sub(r"\(.*?\)", "", name)

    # Thu gọn khoảng trắng
    name = " ".join(name.split())
    return name

def build_map(input_list):
    """
    Tạo map canonical -> list các tiêu đề gốc.
    Bảo toàn phim/người trùng tên.
    """
    res_map = {}
    for title in input_list:
        if not title or not title.strip():
            continue

        key = canonical_title(title)
        # if key != title:
        #     print('test build map: ',title, '==>' ,key)
        if key not in res_map:
            res_map[key] = [ title.strip() ]
        else:
            res_map[key].append(title.strip())


    return res_map


def ner_override_graph(raw_tokens, person_list, film_list):
    """
    Tầng 2 — override tag dựa trên graph bipartite.
    Override NER tags dựa trên bipartite graph.
    Sử dụng longest match để xử lý tên nhiều từ.
    Input: list [(token, tag_raw)]
    Output: list [(token, tag_graph_fixed)]
    """
    '''
    raw_tokens:
        Tầng 1 (raw BIO):
        [('Trấn Thành', 'O'), ('đóng', 'O'), ('trong', 'O'), ('phim', 'O'), ('Bố Già', 'I-LOC')]
    person_list: ['Trấn Thành', 'Ninh Dương Lan Ngọc', 'Ngô Thanh Vân', 'Hồng Đào', 'Kiều Minh Tuấn', 'Victor Vũ',
    film_list: ['Nhà bà Nữ', 'Hai Phượng', 'Tèo em', 'Mùi ngò gai', 'Cổng mặt trời (phim truyền hình)',
    film_set: ... 'Những công dân tập thể', 'Mắt biếc', 'Một lần đi bụi', 'Con đường sáng', 'Bước nhảy hoàn vũ', 'Ngôi nhà trong hẻm', 'Bố già', 'Hello cô Ba', 'Những người đã hết thời', 'Đảo của dân ngụ cư'
    person_set: {'Công Hậu', 'Nikki Dương Nhật Vi', 'Trương Minh Cường', 'Nguyệt Nhi', 'Trí Tuệ', 'Nguyên Trinh', 'Thân Thanh Giang', 'Khoa', 'Nguyễn Thị Tuyết', 'Thanh Ngọc', 'Hoàng Trinh diễn viên', 'Thành Trí', 'Tiến Thành', 'Hoàng Mèo', 'Long Điền', 'Linh Chi', 'Minh Luân', 'Thanh Thúy', 'Kathy Tiên', 'Thảo Quyên', 'Quách Ngọc Tuyên', 'Isaac ca sĩ',

    output hàm: [('Trấn Thành', 'B-PER'), ('đóng', 'B-PER'), ('trong', 'O'), ('phim', 'O'), ('Bố Già', 'I-LOC')]

    print(normalize("Bố già") ) bố già
    '''
    # Chuẩn hóa lookup: tạo set
      # ----- PERSON/ FILM MAP + SET -----
    person_map = build_map(person_list)
    film_map = build_map(film_list)

    person_set = set(normalize_text_for_nlp(k) for k in person_map.keys())
    film_set   = set(normalize_text_for_nlp(k) for k in film_map.keys())

    # thêm cả bản đầy đủ (bố già (phim 2021)) vào set (trc đó set chỉ có bố già)
    for key, arr in film_map.items():
        for full in arr:
            film_set.add(normalize_text_for_nlp(full))

    for k, arr in person_map.items():
        for full in arr:
            person_set.add(normalize_text_for_nlp(full))

    out = []
    i = 0

    while i < len(raw_tokens):
        matched = False
        best_match_length = 0
        best_match_type = None

        # Thử match từ dài nhất (5 từ) xuống 1 từ
        for length in range(min(5, len(raw_tokens) - i), 0, -1):
            phrase_tokens = [raw_tokens[i + j][0] for j in range(length)]
            phrase = " ".join(phrase_tokens)
            phrase_norm = normalize_text_for_nlp(phrase)
            # Bỏ qua nếu phrase rỗng hoặc là noise
            if not phrase_norm:
                continue
            # Check person match
            if phrase_norm in person_set:
                best_match_length = length
                best_match_type = "PER"
                matched = True
                break  # Tìm được match dài nhất, dừng ngay

            # Check film match
            if phrase_norm in film_set:
                best_match_length = length
                best_match_type = "FILM"
                matched = True
                break

        # Nếu match được, gán tag mới
        if matched and best_match_length > 0:
            phrase_tokens = [raw_tokens[i + j][0] for j in range(best_match_length)]
            for j in range(best_match_length):
                prefix = "B" if j == 0 else "I"
                out.append((phrase_tokens[j], f"{prefix}-{best_match_type}"))
            i += best_match_length
        else:
            # Không match → giữ nguyên tag gốc
            out.append(raw_tokens[i])
            i += 1

    return out


# ====================================
# LAYER 3: WIKI ENRICHMENT
# Keywords để detect entity type từ Wiki
PERSON_KEYWORDS = ["diễn viên", "đạo diễn", "ca sĩ", "nghệ sĩ", "mc", "nhà sản xuất"]
FILM_KEYWORDS = ["phim", "bộ phim", "tác phẩm điện ảnh"]

def detect_entity_type_from_wiki(entity_text, wiki_enrich):
    """
    Phát hiện loại entity từ Wikipedia summary.

    Returns: "PER" | "FILM" | None
    """
    entity_norm = normalize_text_for_nlp(entity_text)
    if entity_norm not in wiki_enrich:
        return None

    wiki_data = wiki_enrich[entity_norm]
    summary = str(wiki_data.get("summary", "")).lower()
    if not summary:
        return None
    # Check PERSON (ưu tiên cao hơn)
    if any(kw in summary for kw in PERSON_KEYWORDS):
        return "PER"

    # Check FILM
    if any(kw in summary for kw in FILM_KEYWORDS):
        return "FILM"

    return None


def ner_override_wiki(tokens_after_graph, wiki_enrich_norm):
    if not wiki_enrich_norm:
        return tokens_after_graph
    # # Chuẩn hóa wiki dict
    # wiki_norm = {clean_token(k).lower(): v for k, v in wiki_enrich_norm.items()}

    out = []
    i=0
    while i < len(tokens_after_graph):
        matched = False
        best_match_length = 0
        best_match_type = None
        # Thử match từ dài nhất xuống 1 từ
        for length in range(min(5, len(tokens_after_graph) - i), 0, -1):
            phrase_tokens = [tokens_after_graph[i + j][0] for j in range(length)]
            phrase = " ".join(phrase_tokens)
            phrase_norm = normalize_text_for_nlp(phrase).lower()

            if not phrase_norm:
                continue

            entity_type = detect_entity_type_from_wiki(phrase, wiki_enrich_norm)

            if entity_type:
                best_match_length = length
                best_match_type = entity_type
                matched = True
                break

        # Nếu match được, gán tag mới
        if matched and best_match_length > 0:
            phrase_tokens = [tokens_after_graph[i + j][0] for j in range(best_match_length)]
            for j in range(best_match_length):
                prefix = "B" if j == 0 else "I"
                out.append((phrase_tokens[j], f"{prefix}-{best_match_type}"))
            i += best_match_length
        else:
            # Không match → giữ nguyên
            out.append(tokens_after_graph[i])
            i += 1

    return out


def extract_entities_from_bio(ner_output):

    entities = []
    current_tokens = []
    current_tag = None  # 'B-LOC' -> convert to 'LOC'
    for token, tag in ner_output:
        token = str(token).strip()
        tag = str(tag).strip().upper()

        if not token:
            continue

        # Tag = O → kết thúc entity hiện tại
        if tag == "O":
            if current_tokens:
                entities.append((" ".join(current_tokens), current_tag))
                current_tokens = []
                current_tag = None
            continue

        # Parse BIO tag
        if tag.startswith("B-"):
            # Flush entity cũ nếu có
            if current_tokens:
                entities.append((" ".join(current_tokens), current_tag))

            # Bắt đầu entity mới
            base_tag = tag[2:]  # B-PER → PER
            current_tokens = [token]
            current_tag = base_tag

        elif tag.startswith("I-"):
            base_tag = tag[2:]  # I-PER → PER

            # Nếu I-tag khớp với current tag → tiếp tục entity
            if current_tag == base_tag:
                current_tokens.append(token)
            else:
                # I-tag không khớp → bắt đầu entity mới
                if current_tokens:
                    entities.append((" ".join(current_tokens), current_tag))
                current_tokens = [token]
                current_tag = base_tag

        else:
            # Tag không phải B-/I-/O → xử lý như B-
            if current_tokens:
                entities.append((" ".join(current_tokens), current_tag))
            current_tokens = [token]
            current_tag = tag

    # Flush entity cuối cùng
    if current_tokens:
        entities.append((" ".join(current_tokens), current_tag))

    return entities


# ROLE EXTRACTION
# detect ROLE
ROLE_MAP = {
    # ---- ACTOR ----
    "diễn viên": "actor",
    "nghệ sĩ": "actor",

    # ---- DIRECTOR ----
    "đạo diễn": "director",

    # ---- PRODUCER ----
    "nhà sản xuất": "producer",

    # ---- SCREENWRITER ----
    "biên kịch": "screenwriter",

    # ---- MC / HOST ----
    "mc": "mc",
    "m.c": "mc",
    "người dẫn chương trình": "mc",
    "dẫn chương trình": "mc",
    "host": "mc",

    # ---- COMEDIAN ----
    "hài": "comedian",
    "nghệ sĩ hài": "comedian",

    # ---- FILMMAKER ----
    "nhà làm phim": "filmmaker",
    "movie maker": "filmmaker",
    "film maker": "filmmaker",

    # ---- SINGER ----
    "ca sĩ": "singer",
}

# --- TRÍCH XUẤT TỪ NGỮ CẢNH ---
def extract_roles_from_context(text, entity_name, window_chars=40):
    """
    Tìm role dựa trên từ khóa xuất hiện xung quanh entity trong câu gốc.
    Hỗ trợ tìm cả trước (prefix) và sau (suffix).
    """
    if not text or not entity_name:
        return []

    text_norm = normalize_text_for_nlp(text)
    entity_norm = normalize_text_for_nlp(entity_name)

    roles = set()

    # Tìm vị trí entity trong câu
    start_idx = text_norm.find(entity_norm)
    if start_idx == -1:
        return []

    end_idx = start_idx + len(entity_norm)

    # Lấy vùng văn bản xung quanh (trước và sau entity)
    # Ví dụ: "... [Diễn viên chính] Trấn Thành..." hoặc "...Galaxy Studio [sản xuất]..."
    window_start = max(0, start_idx - window_chars)
    window_end = min(len(text_norm), end_idx + window_chars)

    context_snippet = text_norm[window_start:window_end]

    # Quét Role Map trong vùng context này
    for keyword, role in ROLE_MAP.items():
        # Dùng regex \b để tránh bắt nhầm (ví dụ tránh bắt 'nam' trong 'nam nam')
        # Nhưng tiếng Việt từ ghép nên check in string đơn giản thường hiệu quả hơn
        if keyword in context_snippet:
            roles.add(role)

    return sorted(list(roles))

def extract_roles_from_graph(person_name, bipartite_graph):
    # Trích xuất vai trò từ bipartite graph.
    if not bipartite_graph:
        return []

    person_norm = normalize_text_for_nlp(person_name)
    roles = set()
    # --- Tìm node theo info["name"], không phải key ---
    node_key = None
    for key in bipartite_graph.nodes:
        if normalize_text_for_nlp(key) == person_norm:
            node_key = key
            break

    if not node_key:
        return []

    # --- Lấy OCCUPATION ---
    # LẤY OCCUPATION TỪ NODES VÀ MAP SANG ENGLISH
    node_data = bipartite_graph.nodes[node_key]
    person_info = node_data.get("info", {})
    occupation = person_info.get("occupation", "")
    if occupation:
        # occupation là chuỗi phân tách bằng dấu phẩy
        for item in occupation.split(","):
            item = item.strip().lower()
            if item:
                # Map sang English nếu khớp ROLE_MAP
                mapped = next((role for kw, role in ROLE_MAP.items() if kw in item), item)
                roles.add(mapped)

    # BỔ SUNG TỪ ROLE TRONG EDGES

    for neighbor, info in bipartite_graph[node_key].items():
        if not isinstance(info, dict):
            continue
        role_value = info.get("role", "").lower()
        if not role_value:
            continue
        # if role_value in ["family", "relative"]:
        #     continue
        # Map sang English chuẩn
        roles.add(role_value)
    return sorted(list(roles))


def extract_roles_from_wiki(person_name, wiki_enrich):
    # Trích xuất vai trò từ Wikipedia summary
    if not wiki_enrich:
        return []
    person_norm = normalize_text_for_nlp(person_name)
    roles = set()

    # Tìm key tương ứng
    entry = None
    for key, val in wiki_enrich.items():
        if normalize_text_for_nlp(key) == person_norm:
            entry = val
            break
    if not entry:
        return []
    summary = entry.get("summary", "").lower()
    for keyword, role in ROLE_MAP.items():
        if keyword in summary:
            roles.add(role)

    return sorted(list(roles))


# TỔNG HỢP ROLE TỪ WIKI LẪN GRAPH
def extract_all_roles(text, person_name, bipartite_graph, wiki_enrich):
    # Thêm tham số 'text' vào đầu vào để chạy Context Extraction

    # Kết hợp roles từ cả graph và wiki
    roles_graph = extract_roles_from_graph(person_name, bipartite_graph)
    roles_wiki  = extract_roles_from_wiki(person_name, wiki_enrich)
    # Từ Ngữ cảnh (Câu văn hiện tại)
    roles_context = extract_roles_from_context(text, person_name)
    # Gộp tất cả (Set để loại trùng)
    all_roles = set(roles_graph + roles_wiki + roles_context)
    return sorted(all_roles)

# MAIN NER PIPELINE
# fix TỔ CHỨC
ORG_HINTS = ["studio", "company", "pictures", "production", "entertainment", "corp", "ltd"]

def simple_org_fix(entity):
    name = entity["name"].lower()
    for hint in ORG_HINTS:
        if hint in name:
            entity["type"] = "ORG"
            return entity
    return entity

# COMBINED NER
def run_combine_ner(text, person_list, film_list, wiki_enrich, bipartite_graph):
    # Pipeline NER hoàn chỉnh 3 tầng.
    if not text or not text.strip():
        return []

    # Layer 1: Base NER
    stage1  = ner_raw_underthesea(text)

    # Layer 2: Graph override
    stage2 = ner_override_graph(stage1, person_list, film_list)

    # Layer 3: Wiki override
    # wiki_enrich_lower = { k.lower(): k for k in wiki_enrich }
    wiki_enrich_norm = { normalize_text_for_nlp(k): v for k, v in wiki_enrich.items() }
    stage3 = ner_override_wiki(stage2, wiki_enrich_norm)

    # Cuối cùng: gom lại theo BIO để ra entity
    entities = extract_entities_from_bio(stage3)
    # Enrich với roles cho PERSON
    results = []

    for entity_name, entity_type in entities:
        if entity_type == "O":
            continue
        result = {
            "name": entity_name,
            "type": entity_type,
            "roles": []
        }
        # Extract roles nếu là PERSON
        if entity_type == "PER":
            result["roles"] = extract_all_roles(
                text,
                entity_name,
                bipartite_graph,
                wiki_enrich_norm
            )

        results.append(result)
    results = [simple_org_fix(ent) for ent in results]
    return results


# DEBUG & TESTING

# test
def debug_ner_pipeline(text):
    print("==== INPUT ====")
    print(text)

    stage1 = ner_raw_underthesea(text)
    print("\nTầng 1 (raw BIO):")
    print(stage1)

    stage2 = ner_override_graph(stage1, person_list, film_list)
    print("\nTầng 2 (graph override):")
    print(stage2)

    stage3 = ner_override_wiki(stage2, wiki_enrich)
    print("\nTầng 3 (wiki override):")
    print(stage3)

    final_entities = extract_entities_from_bio(stage3)
    print("\nEntities cuối cùng (sau BIO grouping):")
    print(final_entities)

print('*****************************************************************')
print('TEST 3 TẦNG NER')
debug_ner_pipeline("Trấn Thành đóng trong phim Bố Già cùng các diễn viên khác như ninh dương lan ngọc, kiều minh tuấn")
print('*****************************************************************')


def extract_location_entities(entities):
    return [normalize_entity_name(e["name"]) for e in entities if e["type"] == "LOC"]

def extract_person_entities(entities):
    return [normalize_entity_name(e["name"]) for e in entities if e["type"] == "PER"]

def extract_org_entities(entities):
    return [normalize_entity_name(e["name"]) for e in entities if e["type"] == "ORG"]

def extract_film_entities(entities):
    return [normalize_entity_name(e["name"]) for e in entities if e["type"] == "FILM"]


def tokenize_and_pos_tag(text):

    if not text or not text.strip():
        return []

    # normalize Unicode để tránh lỗi dấu
    text_norm = unicodedata.normalize("NFC", text.strip())

    try:
        pos_out = uts_pos_tag(text_norm)
    except Exception:
        return []

    results = []
    for token, tag in pos_out:
        tok = str(token).strip()
        tg = str(tag).strip().upper()

        # Bỏ các token vô nghĩa
        if not tok:
            continue
        if len(tok) == 1 and tok in ",.!?;:-_\"'()[]{}*/":
            continue

        # Normalize token (bạn có thể tùy chỉnh)
        tok_norm = unicodedata.normalize("NFC", tok)

        results.append((tok_norm, tg))
    return results


def extract_keywords_from_pos(
    pos_output,
    films, persons, orgs, locations,
    min_len = 2,
    stopwords = None
):

    if stopwords is None:
        stopwords = DEFAULT_STOPWORDS
    stopwords = set(w.lower() for w in stopwords)

    # OrderedDict để ghi nhớ thứ tự xuất hiện đầu tiên
    order = OrderedDict()
    counter = Counter()

    for word, pos in pos_output:
        # chuẩn hóa unicode + loại khoảng trắng
        token = unicodedata.normalize("NFKC", word).strip()
        if not token:
            continue
        pos_u = pos.upper()

        if not (pos_u.startswith("N") or pos_u.startswith("A")):
            continue

        # keyword extraction không cần giữ nguyên tên riêng, không cần bảo tồn viết hoa, cũng không cần entity format
        token_clean = normalize_text_for_nlp(token)
        if not token_clean: continue

        # loại stopwords
        if token_clean in stopwords: continue

        # loại từ 1 ký tự
        if len(token_clean) < min_len:continue

        # loại token rác: chỉ toàn số, toàn punctuation, toàn ký tự không phải chữ
        # nhưng ko loại từ có dấu gạch (-), dấu nháy (')
        if re.fullmatch(r"^[\W\d_]+$", token_clean): continue

        # save thứ tự xuất hiện (nếu chưa có)
        if token_clean not in order: order[token_clean] = None
        counter[token_clean] += 1

    # Sort theo tần suất giảm → nếu bằng nhau, theo thứ tự xuất hiện
    keywords = sorted(counter.keys(), key=lambda k: (-counter[k], list(order.keys()).index(k)))

    # Loại token thuộc thực thể NER (film, person, org, loc)
    all_ner = set([*films, *persons, *orgs, *locations])
    all_ner_norm = {unicodedata.normalize("NFKC", x).lower().strip() for x in all_ner}

    # Tách tất cả tokens trong entity đa từ (vd: "bố già" → {"bố", "già"})
    ner_subtokens = set()
    for ent in all_ner_norm:
        for t in ent.split():
            ner_subtokens.add(t.strip())

    # Loại token nếu nó là 1 phần của NER nhiều từ
    clean_keywords = []
    for kw in keywords:
        kw_norm = normalize_text_for_nlp(kw)
        if kw_norm in ner_subtokens:
            continue  # loại "bố", "già", "thành", "lan", "ngọc"...
        clean_keywords.append(kw)

    keywords = clean_keywords


    return keywords



def extract_top_keywords(text, k= 10,stopwords = DEFAULT_STOPWORDS ):

    if not text or not text.strip():
        return []

    # chuẩn hóa unicode trước khi tách từ
    text = unicodedata.normalize("NFKC", text)

    stopwords = {unicodedata.normalize("NFKC", w).lower().strip() for w in stopwords}

    # token hóa (underthesea.word_tokenize giữ token tiếng việt tốt)
    tokens = uts_word_tokenize(text)
    cleaned = []
    for tok in tokens:
        # chuẩn hóa token
        w = unicodedata.normalize("NFKC", tok)
        w = normalize_text_for_nlp(w)

        if not w:
            continue
        if w in stopwords:
            continue
        if len(w) < 2:
            continue
        # loại token rác (toàn số/ký hiệu), nhưng cho phép từ có '-' hoặc ' hoặc /
        # giữ trấn-thành
        if re.fullmatch(r"^[\W\d_]+$", w):
            continue

        cleaned.append(w)

        tok2 = normalize_text_for_nlp(tok)
        if not tok2:
            continue
        if tok2 in stopwords:
            continue
        # loại bỏ token chỉ số/punctuation
        if re.fullmatch(r"[\d\W_]+", tok2):
            continue
        if len(tok2) < 2:
            continue
        cleaned.append(tok2)

    freq = Counter(cleaned)
    return freq.most_common(k)


def generate_topic_nodes(keywords,top_n = None):

    # đảm bảo iterable → list (để cắt top_n)
    kws = list(keywords)
    if top_n is not None:
        kws = kws[:top_n]

    nodes = []
    seen = set()
    for item in kws:
        # hỗ trợ dạng (keyword, count)
        if isinstance(item, (tuple, list)) and len(item) >= 1:
            k = item[0]
        else:
            k = item

        if not isinstance(k, str): k = str(k)

        # chuẩn hóa unicode + strip
        name = unicodedata.normalize("NFKC", k).strip()
        if not name:continue

        # loại các token rác: toàn ký hiệu hoặc toàn số
        if re.fullmatch(r"[\W\d_]+", name): continue
        # collapse spaces
        name_clean  = re.sub(r"\s+", " ", name).lower()
        # chuẩn hóa key kiểm tra trùng, ko dùng để hiển thị
        keynorm = name_clean.lower()
        if keynorm in seen:
            continue # bước loại trùng, nếu trùng thì k làm bc sau
        seen.add(keynorm)

        nodes.append((name_clean, "Topic"))
    return nodes

#======================================================
def add_node(name_raw, typ_raw,nodes,seen):
    name = normalize_entity_name(name_raw)
    if not name:
        return nodes, seen

    typ = normalize_type(typ_raw, default="Node")

    key = name.lower()
    if key in seen:
        return nodes, seen
    seen.add(key)
    nodes.append((name, typ))
    return nodes,seen


def create_new_nodes(films, persons,orgs,locations,topics):
    nodes= []
    seen = set()
    # --- Films ---
    for f in (films or []):
        nodes, seen = add_node(f, "Film", nodes, seen)

    # --- Locations ---
    for loc in (locations or []):
        nodes, seen =add_node(loc, "Location", nodes,seen)


    # --- Topics ---
    for item in (topics or []):
        if isinstance(item, (tuple, list)) and len(item) >= 1:
            name = item[0]
            type_or_freq = item[1] if len(item) > 1 else "Topic"

            # nếu t[1] là số ⇒ hiểu là freq ⇒ set type = Topic
            if isinstance(type_or_freq, (int, float)):
                nodes, seen =add_node(name, "Topic",nodes,seen)
            else:
                nodes,seen=add_node(name, type_or_freq,nodes,seen)
        else:
            nodes,seen=add_node(item, "Topic",nodes,seen)

    # --- Persons ---
    for p in (persons or []):
        nodes,seen=add_node(p, "Person",nodes,seen)

    # --- Organizations ---
    for o in (orgs or []):
        nodes,seen=add_node(o, "Organization",nodes,seen)

    return nodes

def pipeline_extract_nodes_from_summary(summary_text,person_list, film_list, wiki_enrich, B,top_k_keywords=5,stopwords=None):

    # --- Trường hợp input rỗng ---
    if not summary_text or not summary_text.strip():
        return []
    # --- Bước 1: NER ---
    ner_out = run_combine_ner(summary_text,person_list, film_list, wiki_enrich, B)
    films    = extract_film_entities(ner_out)
    persons  = extract_person_entities(ner_out)
    orgs     = extract_org_entities(ner_out)
    locations = extract_location_entities(ner_out)



    # --- Bước 2: POS tagging & keyword extraction ---
    pos_out = tokenize_and_pos_tag(summary_text)
    keywords_by_pos = extract_keywords_from_pos(pos_out,
                                                films, persons, orgs, locations,
                                                stopwords=stopwords)
    # Giới hạn số lượng chủ đề
    # Lấy top k theo pos (nếu enumerate)
    top_keywords = keywords_by_pos[:top_k_keywords]
    # --- Bước 3: Tạo Topic nodes ---
    topic_nodes = generate_topic_nodes(top_keywords)
    # --- Bước 4: Gộp tất cả node --
    new_nodes = create_new_nodes(films=films,
                                 persons=persons,
                                 orgs=orgs,
                                 locations =locations,
                                 topics=topic_nodes
                                 )
    return new_nodes



if __name__ == "__main__":

    sample = "Bố Già là một phim điện ảnh chủ đề gia đình, hài kịch, bối cảnh tại TP.HCM. Diễn viên chính: Trấn Thành, Ninh Dương Lan Ngọc, kiều minh tuấn. Bộ phim do Galaxy Studio sản xuất."

    from underthesea import ner



    ner_raw = ner(sample)


    combine_ner = run_combine_ner(sample,person_list, film_list, wiki_enrich, B)
    print('@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@')
    print("NER raw:", combine_ner)



/usr/local/lib/python3.12/dist-packages/networkx/utils/backends.py:1463: DeprecationWarning: Keyword argument 'link' is deprecated; use 'edges' instead
  return self.orig_func(*args, **kwargs)


Hoa Mặt Trời
hoa mặt trời
Ninh Dương Lan Ngọc
*****************************************************************
TEST 3 TẦNG NER
==== INPUT ====
Trấn Thành đóng trong phim Bố Già cùng các diễn viên khác như ninh dương lan ngọc, kiều minh tuấn

Tầng 1 (raw BIO):
[('Trấn Thành', 'O'), ('đóng', 'O'), ('trong', 'O'), ('phim', 'O'), ('Bố Già', 'I-LOC'), ('cùng', 'O'), ('các', 'O'), ('diễn viên', 'O'), ('khác', 'O'), ('như', 'O'), ('ninh dương', 'O'), ('lan ngọc', 'O'), (',', 'O'), ('kiều minh', 'B-LOC'), ('tuấn', 'I-LOC')]

Tầng 2 (graph override):
[('Trấn Thành', 'B-PER'), ('đóng', 'O'), ('trong', 'O'), ('phim', 'O'), ('Bố Già', 'B-FILM'), ('cùng', 'O'), ('các', 'O'), ('diễn viên', 'O'), ('khác', 'O'), ('như', 'O'), ('ninh dương', 'B-PER'), ('lan ngọc', 'I-PER'), (',', 'I-PER'), ('kiều minh', 'B-PER'), ('tuấn', 'I-PER')]

Tầng 3 (wiki override):
[('Trấn Thành', 'B-PER'), ('đóng', 'O'), ('trong', 'O'), ('phim', 'O'), ('Bố Già', 'B-FILM'), ('cùng', 'O'), ('các', 'O'), ('diễn viên', 'O'), ('kh

In [6]:
print(len(wiki_enrich))
print(wiki_enrich.keys())
print(wiki_enrich['Trấn Thành'].keys())

1764
dict_keys(['Trấn Thành', 'Ninh Dương Lan Ngọc', 'Ngô Thanh Vân', 'Hồng Đào', 'Kiều Minh Tuấn', 'Victor Vũ', 'Charlie Nguyễn', 'Vũ Ngọc Đãng', 'Kaity Nguyễn', 'Jun Vũ', 'Mạnh Trường', 'Lê Bê La', 'Nguyệt Ánh', 'Tú Vi', 'Kha Ly', 'Lương Thế Thành', 'Hùng Thuận', 'Hòa Hiệp (diễn viên)', 'Đình Hiếu', 'Ánh Hoa', 'Huỳnh Anh Tuấn', 'Kinh Quốc', 'Công Ninh', 'Phùng Ngọc Huy', 'Mai Phương (diễn viên)', 'Thiên Kim (diễn viên)', 'Phi Điểu', 'Hoàng Lan (nghệ sĩ)', 'Hải Lý', 'Huy Cường', 'Kim Phượng', 'Hoài An', 'Minh Cường', 'Hồng Thy', 'Khương Ngọc', 'Thanh Trúc', 'Kim Xuân', 'Trọng Nhân', 'Diệu Đức', 'Mã Trung', 'Lộc Uyển', 'Vân Trang', 'Thiên Hương', 'Bảo Thy', 'Yến Trang', 'Hoài Linh', 'Tấn Beo', 'Chí Tài', 'Mạnh Tràng', 'Hiếu Hiền', 'Khổng Tú Quỳnh', 'Kim Thư', 'Ưng Hoàng Phúc', 'Đông Nhi', 'Noo Phước Thịnh', 'Ngô Kiến Huy', 'Phước Sang', 'Tuyết Thu', 'Bảo Quốc', 'Khánh Nam', 'Cát Phượng', 'Phi Phụng', 'Phương Trinh Jolie', 'Tam Thanh', 'Tấn Bo', 'Thành Nam', 'Dương Cẩm Lynh', 'Thanh Ngọ

# tiền xử lý

In [7]:

# loại bỏ dấu câu trước khi xử lý văn bản
RE_PUNCT = re.compile(r"[\u2000-\u206F\u2E00-\u2E7F\'\"“”‘’!#$%&()*+,\-./:;<=>?@\[\]^_`{|}~…—·]+")
# chuẩn hóa khoảng trắng
RE_SPACE = re.compile(r"\s+")

def remove_footnotes(text):
    return re.sub(r'\s*\[\d+\]\s*', ' ', text)
# vào str -> trả str
def split_text_into_sentences(text):
    return underthesea.sent_tokenize(text)


def remove_text_in_parentheses(text):   # xóa text trong ngoặc
    cleaned_text = re.sub(r"\s*\(.*?\)", "", text)  # kết quả sau khi xóa
    return cleaned_text
print(remove_text_in_parentheses('Hoa Mặt Trời (phim truyền hình)')) # ==> Hoa Mặt Trời
# ************************************ Dùng cho văn bản dài

def normalize_text_for_nlp(text):
    """
    Chuẩn hóa toàn diện cho text:
    - Unicode NFKC
    - lowercase
    - remove punctuation
    - remove extra spaces
    - strip
    """
    if not text:
        return ""

    # Normalize Unicode + lowercase
    text = unicodedata.normalize("NFKC", str(text)).lower()

    # Remove punctuation
    text = RE_PUNCT.sub(" ", text)

    # Remove extra spaces
    text = RE_SPACE.sub(" ", text)

    return text.strip()

print(normalize_text_for_nlp('Hoa Mặt Trời')) # ==> hoa mặt trời


import re
import unicodedata

def normalize_entity_name(x):
    # Ninh Dương Lan Ngọc . ==> Ninh Dương Lan Ngọc
    """Chuẩn hoá tên entity: unicode, xoá khoảng trắng, gom space, bỏ dấu câu đầu/cuối."""
    if not isinstance(x, str):
        x = str(x)

    # Chuẩn hoá unicode + bỏ khoảng trắng đầu/cuối
    x = unicodedata.normalize("NFKC", x).strip()
    if not x:
        return ""

    # Nếu chuỗi chỉ toàn ký tự đặc biệt / số → loại bỏ
    if re.fullmatch(r"[\W\d_]+", x):
        return ""

    # Gom nhiều khoảng trắng thành 1
    x = re.sub(r"\s+", " ", x).strip()

    # Hàm kiểm tra ký tự có phải dấu câu (Unicode) không
    # Ví dụ: ., , : ; … “ ” !
    def _is_punct(ch):
        return unicodedata.category(ch).startswith("P")

    # Bỏ dấu câu ở đầu chuỗi
    start = 0
    while start < len(x) and _is_punct(x[start]):
        start += 1

    # Bỏ dấu câu ở cuối chuỗi
    end = len(x) - 1
    while end >= start and _is_punct(x[end]):
        end -= 1

    # Lấy phần còn lại
    x = x[start:end+1].strip()

    # Gom space lại lần cuối
    x = re.sub(r"\s+", " ", x).strip()
    return x


print(normalize_entity_name('Ninh Dương Lan Ngọc .'))



def normalize_type(t, default):
    """Chuẩn hóa type: viết hoa chữ đầu."""
    if not t: return default
    t = str(t).strip()
    if not t: return default
    return t[0].upper() + t[1:]

def norm(s):
    return unicodedata.normalize("NFC", s.strip()).lower()


# Normalize entity ⇒ trả về PER, FILM để khớp với NER combine

def normalize_entity(entity_text, person_list, film_list, wiki_enrich=None):
    if not entity_text:
        return "", "UNK"

    t = entity_text.strip()
    tl = norm(t)

    for p in person_list:
        if tl == norm(p):
            return p, "PER"

    for f in film_list:
        if tl == norm(f):
            return f, "FILM"

    # khớp fuzzy vào wiki
    if wiki_enrich:
        for w in wiki_enrich:
            if tl == norm(w):
                return w, "UNK"

    return t, "UNK"



Hoa Mặt Trời
hoa mặt trời
Ninh Dương Lan Ngọc


# CHẠY MODEL

In [8]:
# Adapter: chuyển output run_combine_ner -> list of spans
def ner_adapter_from_run_combine_ner(text, person_list, film_list, wiki_enrich, bipartite_graph=None):
    """
    Gọi run_combine_ner (của bạn) và trả về danh sách spans:
    [{"start": int, "end": int, "label": "PER"/"FILM"/"ORG"/"LOC", "text": str}, ...]
    """
    ner_out = run_combine_ner(
        text=text,
        person_list=person_list,
        film_list=film_list,
        wiki_enrich=wiki_enrich,
        bipartite_graph=bipartite_graph
    )
    spans = []
    # <-- TÙY vào format run_combine_ner: map fields về start/end/label/text
    # Ví dụ giả định run_combine_ner trả list of (text, label, start, end) hoặc dicts
    # Bạn sửa phần mapping theo đúng output thực tế của run_combine_ner.
    for ent in ner_out:
        # Trường hợp ent là dict như {'text': 'A', 'label': 'PER', 'start': 10, 'end': 11}
        if isinstance(ent, dict) and {"start","end","label","text"} <= set(ent.keys()):
            spans.append({"start": ent["start"], "end": ent["end"], "label": ent["label"], "text": ent["text"]})
        else:
            # fallback: nếu run_combine_ner trả dạng tuple
            try:
                t, lab, st, ed = ent
                spans.append({"start": st, "end": ed, "label": lab, "text": t})
            except Exception:
                # ignore or log
                pass
    return spans


import spacy
nlp = spacy.load("en_core_web_sm")  # hoặc model tiếng vi nếu cần

def ner_adapter_spacy(text):
    doc = nlp(text)
    spans = []
    for ent in doc.ents:
        label = ent.label_
        # Map spaCy labels to your schema if needed, ex: PERSON -> PER
        if label == "PERSON":
            lab = "PER"
        else:
            lab = label  # or map other labels
        spans.append({"start": ent.start_char, "end": ent.end_char, "label": lab, "text": ent.text})
    return spans


def prepare_entity_batch(entity, wiki_enrich, person_list, film_list, ner_model, bipartite_graph=None):
    batch_items = []
    try:
        wiki_text = wiki_enrich.get(entity, {}).get("summary", "")
        if not wiki_text or len(wiki_text) < 50:
            return []
        # gọi ner_model như 1 callable, nó phải trả list of spans như chuẩn ở trên
        ner_out = ner_model(wiki_text) if ner_model is not None else []
        if not ner_out or len(ner_out) < 1:
            return []
        sentences = split_text_into_sentences(wiki_text)
        for sent in sentences:
            s = remove_footnotes(sent)
            s = re.sub(r"\s+", " ", s).strip()
            spans = get_char_spans(s, ner_out)  # giữ nguyên hàm get_char_spans của bạn
            if len(spans) < 2:
                continue
            pairs = permutations(spans, 2)
            for e1, e2 in pairs:
                gap = abs(e1['start'] - e2['start'])
                if gap > 200:
                    continue
                if not (e1["label"] in ["PER","FILM","ORG","LOC"] or
                        e2["label"] in ["PER","FILM","ORG","LOC"]):
                    continue
                masked = create_masked_text(s, e1, e2)
                masked = re.sub(r"\s+", " ", masked).strip()
                batch_items.append({
                    "text": masked,
                    "meta": (e1, e2),
                    "entity_origin": entity
                })
    except Exception as e:
        print(f"⚠ Error preparing {entity}: {e}")
    return batch_items
ner_model = lambda text: ner_adapter_from_run_combine_ner(text, person_list, film_list, wiki_enrich, bipartite_graph=B)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:

import re
import unicodedata
import itertools
from setfit import SetFitModel

# Chỉ thêm entity vào graph_nodes nếu thỏa điều kiện
graph_nodes = []
for e in (person_list + film_list):
    if e in wiki_enrich:
        clean_text = wiki_enrich[e].get("clean_wikitext", "")
        if isinstance(clean_text, str) and clean_text.strip():
            graph_nodes.append(e)





# Regex Char cho tiếng Việt

# VN_WORD_CHAR = r"0-9A-Za-zÀ-ỹđĐ"
# VN_WORD_CHAR = r"A-Za-z0-9À-ỹđĐ"
# VN_WORD_CHAR = r"A-Za-z0-9À-ỿĐđ"
VN_WORD_CHAR = r"A-Za-z0-9À-ỹĐđ"


def get_char_spans(clean_text, ner_out):
    """
    Thay thế hàm convert_ner_to_spans cũ.
    Mục tiêu: Tìm vị trí ký tự (start_char, end_char) của entity trong text
    để sau này chèn thẻ [TAG].
    """
    spans = []
    used_char_ranges = set()

    for ent in ner_out:
        raw_name = ent["name"]
        label = ent["type"] # Ví dụ: PER, ORG...
        # clean_text = normalize_entity_name(raw_name)
        norm_name = normalize_entity_name(raw_name)
        if not norm_name:
            continue

        # Tìm tất cả các vị trí xuất hiện của entity trong text
        pattern = rf"(?<![{VN_WORD_CHAR}])" + re.escape(norm_name) + rf"(?![{VN_WORD_CHAR}])"

        for match in re.finditer(pattern, clean_text, flags=re.IGNORECASE):
            start_char = match.start()
            end_char = match.end()

            # Kiểm tra trùng lặp vị trí (overlap)
            is_overlap = False
            for u_start, u_end in used_char_ranges:
                if not (end_char <= u_start or start_char >= u_end):
                    is_overlap = True
                    break

            if is_overlap:
                continue

            used_char_ranges.add((start_char, end_char))

            spans.append({
                "text": clean_text[start_char:end_char], # Text gốc trong câu
                "start": start_char,
                "end": end_char,
                "label": label
            })

    # Sắp xếp theo vị trí xuất hiện để dễ xử lý sau này
    spans.sort(key=lambda x: x["start"])
    return spans

def create_masked_text(text, span1, span2):
    """
    Hàm quan trọng nhất cho SetFit RE:
    Biến đổi: "Elon Musk mua Twitter"
    Thành: "[PER] Elon Musk [/PER] mua [ORG] Twitter [/ORG]"
    """
    # Xử lý: Luôn thay thế từ thằng nằm sau trước để không làm lệch index thằng nằm trước
    # Sắp xếp 2 span theo thứ tự ngược (thằng nào start lớn hơn thì xử lý trước)
    pair = sorted([span1, span2], key=lambda x: x['start'], reverse=True)

    masked_text = text
    for p in pair:
        # Tạo chuỗi thay thế: [LABEL] text [/LABEL]
        replacement = f"[{p['label']}] {p['text']} [/{p['label']}]"

        # Cắt ghép chuỗi dựa trên index gốc
        masked_text = masked_text[:p['start']] + replacement + masked_text[p['end']:]

    return masked_text

def run_setfit_rel_extraction(
    clean_text,
    ner_out,
    setfit_model, # Truyền model SetFit vào đây
    person_list,
    film_list,
    wiki_enrich=None
):
    # TÁCH VĂN BẢN THÀNH TỪNG CÂU
    sentences = split_text_into_sentences(clean_text)
    triples = []

    for sent in sentences:
        sent = remove_footnotes(sent)
        sent = re.sub(r"\s+", " ", sent).strip()
        # 1. Lấy vị trí các entity (Char Spans)
        entity_spans = get_char_spans(sent, ner_out)


        if len(entity_spans) < 2:
           continue # bỏ qa câu này # Cần ít nhất 2 entity để có quan hệ

        # 2. Tạo các cặp (Pairs) - Permutations (A->B và B->A có thể khác nhau)
        # Nếu quan hệ của là 2 chiều (như married_to), dùng combinations.
        # Nếu quan hệ có hướng (như father_of), dùng permutations.
        pairs = itertools.permutations(entity_spans, 2)
        # pairs = list(itertools.permutations(entity_spans, 2))

        # Chuẩn bị batch input để predict 1 lần cho nhanh
        batch_inputs = []
        batch_meta = [] # Lưu thông tin metadata để map lại kết quả

        for e1, e2 in pairs:

            # --- FILTER 2: cách nhau < 30 từ ---
            gap = abs(e1['start'] - e2['start'])
            if gap > 200:   # tương đương khoảng 30–40 từ
                continue

            # --- FILTER 3: ít nhất 1 trong 2 là person/film ---
            if not (e1["label"] in ["PER", "FILM", 'ORG', 'LOC'] or e2["label"] in ["PER", "FILM",'ORG', 'LOC']):
                continue

            # 3) Mask
            masked_input = create_masked_text(sent, e1, e2)
            masked_input = re.sub(r"\s+", " ", masked_input).strip()

            batch_inputs.append(masked_input)
            batch_meta.append((e1, e2))

        # Không có pair hợp lệ
        if not batch_inputs:
            continue

        # 4) Predict
        predictions = setfit_model.predict(batch_inputs)

        # 5) Map output
        for idx, pred in enumerate(predictions):
            rel = str(pred).upper().strip()
            if rel in ("NO_RELATION", "NONE", "O", ""):
                continue

            e1, e2 = batch_meta[idx]
            subj_raw = normalize_entity_name(e1['text'])
            obj_raw  = normalize_entity_name(e2['text'])

            subj_norm, subj_type = normalize_entity(subj_raw, person_list, film_list, wiki_enrich)
            obj_norm,  obj_type  = normalize_entity(obj_raw,  person_list, film_list, wiki_enrich)

            # --- FILTER UNKNOWN chỉ nếu không phải person/film ---
            if subj_type == "Unknown" and subj_norm not in film_list + person_list:
                continue
            if obj_type == "Unknown" and obj_norm not in film_list + person_list:
                continue

            # tránh self-loop
            if subj_norm == obj_norm:
                continue
            # --- NEW FILTER 4: Check type compatibility ---
            if not is_valid_pair(subj_type, obj_type, rel):
                # print("Invalid type pair:", (subj_type, obj_type), "for relation", rel)
                continue
            triples.append((subj_norm, pred, obj_norm))

    # unique
    triples = list(dict.fromkeys(triples))
    return triples

def run_setfit_rel_extraction_debug(
    clean_text,
    ner_out,
    setfit_model,
    person_list,
    film_list,
    wiki_enrich=None,
    debug=False
):

    if debug:
        print("\n===== DEBUG WITH SENTENCE SPLIT =====\n")

    # ================================================================
    # COMMENT IN HOA: TÁCH VĂN BẢN THÀNH TỪNG CÂU
    sentences = split_text_into_sentences(clean_text)
    # ================================================================

    triples = []

    for si, sent in enumerate(sentences):

        if debug:
            print(f"\n--- SENTENCE {si}: {sent}\n")

        sent = remove_footnotes(sent)
        sent = re.sub(r"\s+", " ", sent).strip()

        spans = get_char_spans(sent, ner_out)

        if debug:
            print("Entity spans:", spans)

        if len(spans) < 2:
            continue

        pairs = list(itertools.permutations(spans, 2))
        batch_inputs = []
        meta = []

        for e1, e2 in pairs:

            if abs(e1['start'] - e2['start']) > 200:
                if debug: print("Skip: gap too large")
                continue

            masked = create_masked_text(sent, e1, e2)
            masked = re.sub(r"\s+", " ", masked)

            if debug:
                print("Masked:", masked)

            batch_inputs.append(masked)
            meta.append((e1, e2))

        if not batch_inputs:
            continue

        preds = setfit_model.predict(batch_inputs)

        if debug:
            print("Preds:", preds)

        for idx, pred in enumerate(preds):
            rel = str(pred).upper().strip()
            if rel in ["NO_RELATION", "NONE", "O", ""]:
                if debug: print("Skip no relation")
                continue

            e1, e2 = meta[idx]

            subj_raw = normalize_entity_name(e1['text'])
            obj_raw = normalize_entity_name(e2['text'])

            subj_norm, subj_type = normalize_entity(subj_raw, person_list, film_list, wiki_enrich)
            obj_norm, obj_type = normalize_entity(obj_raw, person_list, film_list, wiki_enrich)

            if not is_valid_pair(subj_type, obj_type, rel):
                if debug: print("Invalid type pair, skip")
                continue

            if subj_norm == obj_norm:
                continue

            triples.append(((subj_norm, subj_type), rel, (obj_norm, obj_type)))
            if debug:
                print("✓ ADD TRIPLE:", ((subj_norm, subj_type), rel, (obj_norm, obj_type)))

    triples = list(dict.fromkeys(triples))

    if debug:
        print("\n======= FINAL TRIPLES =======")
        print(triples)

    return triples



# Set global relation set để dedup toàn cục


def dedup_triples(triples, seen_set=None):
    """
    Dedup triples, có thể dùng set cục bộ hoặc toàn cục
    """
    if seen_set is None:
        seen_set = set()  # Set cục bộ cho mỗi lần gọi

    new = []
    for s, r, o in triples:
        # Đảm bảo s và o là strings, không phải tuples
        if isinstance(s, tuple):
            s_str = s[0] if isinstance(s[0], str) else str(s[0])
        else:
            s_str = str(s)

        if isinstance(o, tuple):
            o_str = o[0] if isinstance(o[0], str) else str(o[0])
        else:
            o_str = str(o)

        key = (s_str, r, o_str)
        if key not in seen_set:
            seen_set.add(key)
            new.append((s, r, o))
    return new

# ==============================================================================
# 5. MAIN PROCESS (Đã cập nhật để dùng SetFit)
# ==============================================================================

# Load SetFit Model (cần train trước và lưu vào folder hoặc dùng path huggingface)
#  đã train trước và lưu model


try:
    # Load model RE
    # Nếu chưa train, có thể comment dòng này lại để test logic code trước
    re_model = SetFitModel.from_pretrained(f"{GG_COLAB}/data/re_model")
    print("SetFit model loaded successfully.")
except Exception as e:
    print("Chưa load được model SetFit (hãy train trước):", e)
    re_model = None

RELATION_TYPES = {
    "ACTED_IN",
    "DIRECTED",
    "SPOUSE_OF",
    "COLLABORATED_WITH",
    "SAME_HOMETOWN_AS",
    "SAME_SCHOOL_AS"
}
VALID_TYPES = {
    "ACTED_IN": [("PER", "FILM")],
    "DIRECTED": [("PER", "FILM")],

    "SPOUSE_OF": [("PER", "PER")],
    "COLLABORATED_WITH": [("PER", "PER")],

    "SAME_HOMETOWN_AS" : [("PER", "PER")],
    "SAME_SCHOOL_AS": [("PER", "PER")],

}

def is_valid_pair(subj_type, obj_type, relation):
    relation = relation.upper()
    valid = VALID_TYPES.get(relation, [])
    return (subj_type, obj_type) in valid



RE_RES_FILE = f"{GG_COLAB}/data/re_res.jsonl"
from tqdm import tqdm

if re_model is not None:

    for entity in tqdm(
        graph_nodes,
        desc="Running RE pipeline",
        unit="entity"
    ):


        if entity not in wiki_enrich:
            continue

        clean_text = wiki_enrich[entity].get("clean_wikitext", "")
        if not clean_text:
            continue


        ner_out = run_combine_ner(
            text=clean_text,
            person_list=person_list,
            film_list=film_list,
            wiki_enrich=wiki_enrich,
            bipartite_graph=B
        )


        relations = run_setfit_rel_extraction_debug(
            clean_text=clean_text,
            ner_out=ner_out,
            setfit_model=re_model,
            person_list=person_list,
            film_list=film_list,
            wiki_enrich=wiki_enrich
        )

        # 3) Lọc quan hệ sai schema (dựa vào VALID_TYPES)
        filtered_relations = []
        for (s, r, o) in relations:
            # s và o là tuple (name, type)
            subj_type = s[1]  # Lấy type từ tuple
            obj_type = o[1]   # Lấy type từ tuple

            if is_valid_pair(subj_type, obj_type, r):
                filtered_relations.append((s, r, o))
        # 4) Dedup
        filtered_relations = dedup_triples(filtered_relations)




        # Mỗi lần chạy toàn pipeline → reset file
        # (Chỉ reset 1 lần ở entity đầu tiên)
        if entity == graph_nodes[0]:
            open(RE_RES_FILE, "w", encoding="utf-8").close()

        # Chuẩn bị danh sách quan hệ cho entity này
        import json
        relations_list = []
        for (s, r, o) in filtered_relations:
            s_name = s[0] if isinstance(s, tuple) else s
            o_name = o[0] if isinstance(o, tuple) else o

            relations_list.append({
                "subject": s_name,
                "relation": r,
                "object": o_name
            })

        # Ghi 1 dòng JSON cho entity
        with open(RE_RES_FILE, "a", encoding="utf-8") as f:
            json.dump({
                "entity": entity,
                "relations": relations_list
            }, f, ensure_ascii=False)
            f.write("\n")

        print(f"Wrote RE for {entity} → {RE_RES_FILE}")

        print("\n************************* RE ********************************")
        if filtered_relations:
            print(f"[{entity}] → {filtered_relations}")
        else:
            print(f"[{entity}] → (no extracted relations)")

else:
    print("Chưa train model SetFit RE!")

The tokenizer you are loading from '/content/drive/MyDrive/MXH/data/re_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


SetFit model loaded successfully.


Running RE pipeline:   0%|          | 1/1764 [00:38<18:48:36, 38.41s/entity]

Wrote RE for Trấn Thành → /content/drive/MyDrive/MXH/data/re_res.jsonl

************************* RE ********************************
[Trấn Thành] → [(('Trấn Thành', 'PER'), 'SPOUSE_OF', ('Hari Won', 'PER')), (('Hari Won', 'PER'), 'SPOUSE_OF', ('Trấn Thành', 'PER')), (('Trấn Thành', 'PER'), 'SAME_SCHOOL_AS', ('Vĩnh Long', 'PER')), (('Vĩnh Long', 'PER'), 'SAME_SCHOOL_AS', ('Trấn Thành', 'PER')), (('Trấn Thành', 'PER'), 'COLLABORATED_WITH', ('Thủy Tiên', 'PER')), (('Thủy Tiên', 'PER'), 'COLLABORATED_WITH', ('Trấn Thành', 'PER')), (('Trấn Thành', 'PER'), 'ACTED_IN', ('Đất phương Nam', 'FILM')), (('Mạc Can', 'PER'), 'ACTED_IN', ('Đất phương Nam', 'FILM'))]


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
Running RE pipeline:   0%|          | 2/1764 [00:45<9:50:12, 20.10s/entity] 

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
Running RE pipeline:  23%|██▎       | 406/1764 [2:58:15<49:20:43, 130.81s/entity]

Wrote RE for Ngọc Thuận → /content/drive/MyDrive/MXH/data/re_res.jsonl

************************* RE ********************************
[Ngọc Thuận] → (no extracted relations)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
Running RE pipeline:  23%|██▎       | 407/1764 [3:00:08<47:12:17, 125.23s/entity]

Wrote RE for Ngô Tuấn → /content/drive/MyDrive/MXH/data/re_res.jsonl

************************* RE ********************************
[Ngô Tuấn] → (no extracted relations)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
